In [ ]:
import cv2, os, numpy as np
clip = "videos/video_vial_fill_team-msoe-20261030T142247Z_20261030_145629_267244.mp4"
out = "frames/145629"; os.makedirs(out, exist_ok=True)
cap, i, k = cv2.VideoCapture(clip), 0, 0
while True:
    ok, f = cap.read()
    if not ok: break
    if i % 3 == 0:
        cv2.imwrite(f"{out}/{k:05d}.jpg", f); k += 1
    i += 1
print(k, "frames saved")

In [ ]:
import matplotlib.pyplot as plt
  for idx in [0, k - 50]:
      img = cv2.imread(f"{out}/{idx:05d}.jpg")[:, :, ::-1]
      plt.figure(figsize=(8, 6)); plt.imshow(img); plt.title(idx)
      plt.xticks(range(0, 640, 40)); plt.yticks(range(0, 480, 40)); plt.grid(alpha=.4)


In [ ]:
import torch
from sam2.sam2_video_predictor import SAM2VideoPredictor
pred = SAM2VideoPredictor.from_pretrained("facebook/sam2.1-hiera-small")
VIAL_BOX   = np.array([280, 60, 350, 425], np.float32)   # x0,y0,x1,y1 on frame 0 – edit from cell 2
LIQ_FRAME  = k - 50                                       # a frame where the vial has liquid
LIQ_POINT  = np.array([[315, 380]], np.float32)           # a pixel inside the liquid – edit
masks = {}
with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
    st = pred.init_state(video_path=out, offload_video_to_cpu=True, offload_state_to_cpu=True)
    pred.add_new_points_or_box(st, frame_idx=0, obj_id=1, box=VIAL_BOX)
    pred.add_new_points_or_box(st, frame_idx=LIQ_FRAME, obj_id=2,
                               points=LIQ_POINT, labels=np.array([1], np.int32))
    for rev in (False, True):
        for f, ids, logits in pred.propagate_in_video(st, reverse=rev):
            for j, oid in enumerate(ids):
                masks[(f, oid)] = (logits[j, 0] > 0).cpu().numpy()

In [ ]:
import pandas as pd
rows = []
for f in range(k):
    liq = masks.get((f, 2))
    img = cv2.imread(f"{out}/{f:05d}.jpg")
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    r = {"frame_10fps": f, "video_frame": f * 3}
    if liq is not None and liq.any():
        ys = np.where(liq.any(1))[0]
        L, a, b = np.median(lab[liq], 0)
        r.update(liq_top_y=ys.min(), liq_bottom_y=ys.max(), liq_area_px=int(liq.sum()),
                 L=L * 100 / 255, a=a - 128, b=b - 128)
    rows.append(r)
df = pd.DataFrame(rows); df.to_csv("sam_145629.csv", index=False); df.plot(x="frame_10fps", y="liq_area_px")

In [ ]:
for f in [0, k // 4, k // 2, 3 * k // 4, k - 1]:
     img = cv2.imread(f"{out}/{f:05d}.jpg")[:, :, ::-1].copy()
     for oid, col in [(1, (0, 255, 0)), (2, (255, 0, 255))]:
         m = masks.get((f, oid))
         if m is not None: img[m] = (0.5 * img[m] + 0.5 * np.array(col)).astype(np.uint8)
     plt.figure(figsize=(5, 4)); plt.imshow(img); plt.title(f)